In [1]:
import pandas as pd

url = "https://huggingface.co/datasets/chuso6/har_features/resolve/main/har_features.csv"

df = pd.read_csv(url)
df.head()

,activity,subject,window_id,T_acc_mean_x,T_acc_mean_y,T_acc_mean_z,T_acc_std_x,T_acc_std_y,T_acc_std_z,T_acc_max_x,...,LL_mag_max_x,LL_mag_max_y,LL_mag_max_z,LL_mag_corr_xy,LL_mag_corr_xz,LL_mag_corr_yz,LL_mag_mag_mean,LL_mag_mag_std,LL_mag_mag_auc,LL_mag_mag_mean_diff
0,1,1,0,8.015509,1.058076,5.553903,0.129444,0.039797,0.191729,8.1605,...,0.74182,0.30267,-0.055365,-0.380922,0.214412,-0.094971,0.800508,0.000745,2.369513,0.000941
1,1,1,1,7.920071,1.126935,5.683707,0.058170,0.026639,0.105984,8.0412,...,0.74320,0.30342,-0.054963,-0.351583,0.448888,-0.306916,0.801040,0.000701,2.371086,0.000761
2,1,1,2,8.001183,1.141395,5.559029,0.095242,0.030720,0.148445,8.1763,...,0.74335,0.30377,-0.054945,-0.231525,0.377849,-0.342104,0.801930,0.000862,2.373707,0.000829
3,1,1,3,7.941989,1.143843,5.658659,0.059360,0.024328,0.094272,8.1160,...,0.74302,0.30397,-0.054711,-0.266598,0.365792,-0.255355,0.802269,0.000731,2.374725,0.000797
4,1,1,4,7.996011,1.138048,5.567233,0.042821,0.021047,0.067826,8.0860,...,0.74316,0.30423,-0.055413,-0.169517,0.621999,-0.270246,0.802356,0.000820,2.375000,0.000938


In [2]:
print("--- Hold-Out Independiente del Sujeto (ADAPTADO) ---")

# Tus sujetos reales
print("Sujetos disponibles:", sorted(df['subject'].unique()))

# Split manual balanceado
sujetos_para_train = [1, 2, 3, 4]
sujetos_para_validacion = [5, 6]
sujetos_para_test = [7, 8]

# Máscaras
mask_train = df['subject'].isin(sujetos_para_train)
mask_val = df['subject'].isin(sujetos_para_validacion)
mask_test = df['subject'].isin(sujetos_para_test)

# Features (quitamos columnas no útiles)
columns_to_drop = ['activity', 'subject', 'window_id']

X_train = df[mask_train].drop(columns=columns_to_drop)
y_train = df[mask_train]['activity']

X_val = df[mask_val].drop(columns=columns_to_drop)
y_val = df[mask_val]['activity']

X_test = df[mask_test].drop(columns=columns_to_drop)
y_test = df[mask_test]['activity']

# Información
print("\nSujetos en Train      :", sorted(df[mask_train]['subject'].unique()))
print("Sujetos en Validación :", sorted(df[mask_val]['subject'].unique()))
print("Sujetos en Test       :", sorted(df[mask_test]['subject'].unique()))

print("\nFormas:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

--- Hold-Out Independiente del Sujeto (ADAPTADO) ---
Sujetos disponibles: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

Sujetos en Train      : [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Sujetos en Validación : [np.int64(5), np.int64(6)]
Sujetos en Test       : [np.int64(7), np.int64(8)]

Formas:
X_train: (7600, 240)
X_val  : (3800, 240)
X_test : (3800, 240)


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

In [4]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, GroupKFold

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ================================
# 1. Definir grupos (SUJETOS)
# ================================
groups = df[df['subject'].isin(sujetos_para_train)]['subject']

# ================================
# 2. Pipeline
# ================================
pipeline_lr = Pipeline([
    ('scaler', RobustScaler()),
    ('classifier', LogisticRegression(random_state=42))
])

# ================================
# 3. Grid de hiperparámetros
# ================================
param_grid = {
    'classifier__penalty': ['l2'],
    'classifier__C': [0.1, 1, 10, 25],
    'classifier__solver': ['lbfgs'],
    'classifier__max_iter': [200, 500, 1000]
}

# ================================
# 4. Validación por sujeto
# ================================
gkf = GroupKFold(n_splits=4)  # porque tienes 4 sujetos en train

search = GridSearchCV(
    pipeline_lr,
    param_grid,
    cv=gkf,
    n_jobs=-1,
    verbose=1
)

# ================================
# 5. Entrenamiento
# ================================
search.fit(X_train, y_train, groups=groups)

print(f"\nMejores parámetros: {search.best_params_}")
print(f"Mejor score CV: {search.best_score_:.4f}")

Fitting 4 folds for each of 12 candidates, totalling 48 fits

Mejores parámetros: {'classifier__C': 1, 'classifier__max_iter': 200, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
Mejor score CV: 0.8684


In [5]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold

# Ocultar advertencias de convergencia para una salida más limpia
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ================================
# 1. Definir grupos (SUJETOS)
# ================================
# 'df' y 'sujetos_para_train' provienen de tus celdas anteriores
groups = df[df['subject'].isin(sujetos_para_train)]['subject']

# ================================
# 2. Pipeline
# ================================
pipeline_mlp = Pipeline([
    ('scaler', RobustScaler()), # Mantenemos el mismo escalador que usaste
    ('classifier', MLPClassifier(
        random_state=42,
        early_stopping=True,    # Detiene el entrenamiento si no hay mejora (ahorra mucho tiempo)
        validation_fraction=0.1 # Usa 10% del train interno para el early stopping
    ))
])

# ================================
# 3. Grid de hiperparámetros
# ================================
# ⚠️ ADVERTENCIA: Este grid generará 24 combinaciones (24 * 4 folds = 96 ajustes).
# El MLP tarda, ten paciencia.
param_grid_mlp = {
    'classifier__hidden_layer_sizes': [(64,), (128,), (64, 32)], # Diferentes arquitecturas de capas/neuronas
    'classifier__activation': ['relu', 'tanh'],                  # Funciones de activación no lineales
    'classifier__alpha': [0.0001, 0.01],                         # Penalización L2 (Regularización)
    'classifier__learning_rate_init': [0.001, 0.01],             # Tasa de aprendizaje inicial
    'classifier__max_iter': [500]                                # Límite fijo (early_stopping cortará antes si es posible)
}

# ================================
# 4. Validación por sujeto
# ================================
# n_splits=4 corresponde a los 4 sujetos que dejaste en el conjunto de train (Sujetos 1, 2, 3 y 4)
gkf = GroupKFold(n_splits=4)

search_mlp = GridSearchCV(
    pipeline_mlp,
    param_grid_mlp,
    cv=gkf,
    n_jobs=-1,  # Utiliza todos los procesadores disponibles
    verbose=2   # verbose=2 imprimirá el progreso para que sepas que Colab no se ha congelado
)

# ================================
# 5. Entrenamiento
# ================================
print("Iniciando búsqueda de hiperparámetros para MLP... (Esto puede tardar unos minutos)")
search_mlp.fit(X_train, y_train, groups=groups)

print(f"\nMejores parámetros para el Perceptrón: {search_mlp.best_params_}")
print(f"Mejor score CV: {search_mlp.best_score_:.4f}")

Iniciando búsqueda de hiperparámetros para MLP... (Esto puede tardar unos minutos)
Fitting 4 folds for each of 24 candidates, totalling 96 fits

Mejores parámetros para el Perceptrón: {'classifier__activation': 'tanh', 'classifier__alpha': 0.01, 'classifier__hidden_layer_sizes': (64, 32), 'classifier__learning_rate_init': 0.001, 'classifier__max_iter': 500}
Mejor score CV: 0.8805


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold

# ================================
# 1. Definir grupos (SUJETOS)
# ================================
# Asume que 'df' y 'sujetos_para_train' ya están en memoria
groups = df[df['subject'].isin(sujetos_para_train)]['subject']

# ================================
# 2. Pipeline
# ================================
# Omitimos el scaler porque Random Forest no lo necesita matemáticamente
pipeline_rf = Pipeline([
    ('classifier', RandomForestClassifier(random_state=42))
])

# ================================
# 3. Grid de hiperparámetros
# ================================
# Grid optimizado para evitar colapsar la RAM de Colab.
# Genera 16 combinaciones x 4 pliegues = 64 ajustes.
param_grid_rf = {
    'classifier__n_estimators': [100, 200],           # Número total de árboles en el bosque
    'classifier__max_depth': [None, 20],              # Profundidad máxima (None = crecen hasta ser puros)
    'classifier__min_samples_split': [2, 5],          # Número mínimo de muestras para dividir un nodo intermedio
    'classifier__min_samples_leaf': [1, 2]            # Número mínimo de muestras que debe tener una hoja final
}

# ================================
# 4. Validación por sujeto
# ================================
gkf = GroupKFold(n_splits=4)

search_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=gkf,
    n_jobs=-1,  # Utiliza todos los núcleos de la CPU de Colab
    verbose=2   # Muestra el progreso en tiempo real
)

# ================================
# 5. Entrenamiento
# ================================
print("Iniciando búsqueda de hiperparámetros para Random Forest...")
search_rf.fit(X_train, y_train, groups=groups)

print(f"\nMejores parámetros para el Bosque Aleatorio: {search_rf.best_params_}")
print(f"Mejor score CV: {search_rf.best_score_:.4f}")

Iniciando búsqueda de hiperparámetros para Random Forest...
Fitting 4 folds for each of 16 candidates, totalling 64 fits

Mejores parámetros para el Bosque Aleatorio: {'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
Mejor score CV: 0.8554
